In [2]:
import pandas as pd 
import numpy as np 
from pathlib import Path
from src.data.functions import json_to_dict

In [3]:
top_dir = Path().absolute()
PARENT_FILE_PATH = top_dir / "models" / "2025-10-06 TSLP Platform Fit Model"
model_config = json_to_dict(Path(PARENT_FILE_PATH, "TSLP_PF_model_parameters.json"))

a_matrix = np.array(model_config["a_matrix"])
b_matrix = np.array(model_config["b_matrix"])

In [4]:
def controllability_matrix(A, B):
    n = A.shape[0]
    C = B
    for i in range(1, n):
        C = np.hstack((C, np.linalg.matrix_power(A, i) @ B))
    return C

In [5]:
C = controllability_matrix(a_matrix, b_matrix)

rank = np.linalg.matrix_rank(C)
print("Rank of controllability matrix:", rank)

if rank == a_matrix.shape[0]:
    print("The system is controllable.")
else:
    print("The system is NOT controllable.")

Rank of controllability matrix: 5
The system is controllable.


In [6]:
U, S, Vt = np.linalg.svd(b_matrix)

print("Singular values:", S)
print("Input direction V[:, 0]:", Vt.T[:, 0])
print("Mapped state direction U[:, 0]:", U[:, 0])

Singular values: [0.20290792 0.20290746 0.20290707]
Input direction V[:, 0]: [ 0.06469873 -0.74481328 -0.66412894]
Mapped state direction U[:, 0]: [ 0.33001871  0.88484489  0.279664   -0.09334245  0.14564478]


In [7]:
def check_stability(A, dt=None):
    lam = np.linalg.eigvals(A)
    if dt is None:
        # continuous
        unstable = [l for l in lam if np.real(l) > 0]
        borderline = [l for l in lam if np.real(l) > -1e-8 and np.real(l) <= 0]
        print("Eigenvalues (continuous):", lam)
        print("Unstable (Re>0):", unstable)
        print("Borderline (Re≈0):", borderline)
    else:
        # discrete (A is discrete-time)
        mags = np.abs(lam)
        unstable = [l for l,m in zip(lam,mags) if m > 1.0]
        borderline = [l for l,m in zip(lam,mags) if np.isclose(m,1.0,rtol=1e-6,atol=1e-8)]
        print("Eigenvalues (discrete):", lam)
        print("Unstable (|λ|>1):", unstable)
        print("Borderline (|λ|≈1):", borderline)
    return lam

eigens = check_stability(a_matrix)

Eigenvalues (continuous): [-0.11369479+0.22302469j -0.11369479-0.22302469j -0.03999555+0.j
 -0.00241865+0.07658523j -0.00241865-0.07658523j]
Unstable (Re>0): []
Borderline (Re≈0): []


In [8]:
import numpy as np
from numpy.linalg import matrix_rank, eigvals, inv, svd

def controllability_matrix(A,B):
    n = A.shape[0]
    mats = [np.linalg.matrix_power(A,i).dot(B) for i in range(n)]
    return np.hstack(mats)

def check_model(A,B,C=None,D=None, model_name="model"):
    n = A.shape[0]
    print(f"\n=== DIAGNOSTICS for {model_name} ===")
    # eigenvalues
    lam = eigvals(A)
    print("A eigenvalues:", lam)
    # controllability
    Ctrb = controllability_matrix(A,B)
    rank_ctrb = matrix_rank(Ctrb)
    print(f"Controllability matrix shape {Ctrb.shape}, rank = {rank_ctrb} (need {n})")
    # B conditioning
    u,s,vh = svd(B, full_matrices=False)
    print("B singular values:", s)
    print("B col norms:", np.linalg.norm(B, axis=0))
    # DC gain (continuous)
    if C is not None:
        try:
            Ainv = inv(-A)
            G0 = C.dot(Ainv).dot(B) + (D if D is not None else 0)
            print("DC gain C(-A)^{-1}B + D:\n", G0)
        except Exception as e:
            print("DC gain compute failed:", e)
    # modal controllability (participation of B in eigenmodes)
    try:
        lam, V = np.linalg.eig(A)
        Vinv = np.linalg.inv(V)
        # modal controllability measure per mode: norm of (Vinv * B) row
        modal_ctrl = [np.linalg.norm(Vinv[i,:].dot(B)) for i in range(n)]
        print("Modal controllability (per eigenmode):", modal_ctrl)
    except Exception as e:
        print("Modal controllability failed:", e)

# Example usage:
# check_model(A1,B1,C=C,D=D, model_name="first_model")
check_model(a_matrix,b_matrix,C=None,D=None, model_name="second_model")

condition_number = np.linalg.cond(np.array(b_matrix))
print("Condition number of the matrix:", condition_number)


=== DIAGNOSTICS for second_model ===
A eigenvalues: [-0.11369479+0.22302469j -0.11369479-0.22302469j -0.03999555+0.j
 -0.00241865+0.07658523j -0.00241865-0.07658523j]
Controllability matrix shape (5, 15), rank = 5 (need 5)
B singular values: [0.20290792 0.20290746 0.20290707]
B col norms: [0.20290729 0.20290763 0.20290753]
Modal controllability (per eigenmode): [np.float64(0.7143002546297635), np.float64(0.7143002546297637), np.float64(0.9123591354114342), np.float64(0.9967789625088899), np.float64(0.99677896250889)]
Condition number of the matrix: 1.0000041754086393
